importing all relevant libraries

In [1]:
import math

import equinox as eqx
import jax
import jax.lax as lax
import jax.numpy as jnp
import jax.random as jrandom
import numpy as np
import optax # pip install optax
import esm  # pip install fair-esm==2.0.0
import esm2quinox
import jax.random as jr
import os
import pandas as pd
from torch.utils.data import  DataLoader,random_split, RandomSampler


Loading our custom dataset

In [9]:
# create custom dataset class
class Seq_AFFVAL_Dataset():
    def __init__(self, data_folder, transform=None, target_transform=None):
        self.data = pd.read_csv(data_folder)
        self.seqs = self.data['sequence'].tolist()
        self.affVals = self.data["mmpbsa"]
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        affVal = self.affVals[idx]
        if self.transform:
            padd_len = 15 - len(seq)
            seq = self.transform([seq + '.' * padd_len])[0]
            
            seq = np.array(seq)
  
        if self.target_transform:
            affVal = self.target_transform(affVal)
        return seq, affVal
    
# create dataset 
Dataset= Seq_AFFVAL_Dataset(data_folder="/home/kunzj/bindcraft_modified/flexs/landscape.csv",transform=esm2quinox.tokenise)
size_data = Dataset.__len__()
#split dataset into training and test set
training_data, test_data =random_split(Dataset,[int(size_data * 0.9),int(size_data * 0.1)])
sampler = RandomSampler(training_data, replacement=True, num_samples=500)
training_DataLoader = DataLoader(training_data,sampler=sampler,batch_size=2)
test_DataLoader = DataLoader(test_data,batch_size=1,shuffle=True)



In [10]:
data_key, model_key = jrandom.split(jrandom.PRNGKey(0), 2)

In [44]:
class AFF_PREDICTOR(eqx.Module):
    model: esm2quinox.ESM2
    mlp: eqx.nn.MLP
    linear: eqx.nn.Linear

    def __init__(self, model, key):
        self.model = model #(num_layers=3, embed_size=32, num_heads=2, token_dropout=False, key=key)
        self.mlp = eqx.nn.MLP(in_size=320,out_size= 1, key=key,width_size=10,depth=5)  # assuming embed_size=33
        self.linear = eqx.nn.Linear(in_features=1, out_features=1, key=key)

    def __call__(self, tokens):
        
        out = self.model(tokens)
        # Pooling: mean over sequence length (ignoring padding)
        # mask = tokens
        # jax.debug.print('tokens {tokens}',tokens=tokens)
        # lengths = mask.sum(axis=1, keepdims=True)
        # pooled = (out.hidden * mask[..., None]).sum(axis=1) / lengths
        out_mlp = self.mlp(out.hidden[0])
        return self.linear(out_mlp)



In [45]:
torch_model, _ = esm.pretrained.esm2_t6_8M_UR50D()
model_esm2 = esm2quinox.from_torch(torch_model)
model_aff = AFF_PREDICTOR(model=model_esm2,key=model_key)

In [48]:
@eqx.filter_value_and_grad
def compute_loss(model, x, y):
    pred_y = jax.vmap(model)(x)
    # trains with MSE
    return ((pred_y - y) ** 2).mean()



@eqx.filter_jit
def make_step(model, x, y, opt_state):

    loss, grads = compute_loss(model, x, y)
    updates, opt_state = optim.update(grads, opt_state)
    model = eqx.apply_updates(model, updates)
    return loss, model, opt_state


optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(model_aff, eqx.is_inexact_array))
for step, (x, y) in zip(range(500), training_DataLoader):
    x=jnp.array(x)
    y=jnp.array(y)
    # print(x.shape)
    loss, model, opt_state = make_step(model_aff, x, y, opt_state)
    loss = loss.item()
    print(f"step={step +1}, loss={"%.2f" % loss}")



step=1, loss=3726.25
step=2, loss=3855.19
step=3, loss=4370.74
step=4, loss=4239.32
step=5, loss=4524.51
step=6, loss=4888.60
step=7, loss=4685.00
step=8, loss=4685.00
step=9, loss=5601.43
step=10, loss=4328.60
step=11, loss=4705.37
step=12, loss=4072.55
step=13, loss=3422.03
step=14, loss=3574.69
step=15, loss=4850.01
step=16, loss=4463.87
step=17, loss=3914.43
step=18, loss=4788.91
step=19, loss=3995.95
step=20, loss=4870.66
step=21, loss=4707.39
step=22, loss=4981.01
step=23, loss=4723.01
step=24, loss=4905.93
step=25, loss=4759.89
step=26, loss=3749.06
step=27, loss=4801.37
step=28, loss=3894.71
step=29, loss=4794.66
step=30, loss=4549.24
step=31, loss=3866.05
step=32, loss=5652.33
step=33, loss=4157.91
step=34, loss=5047.64
step=35, loss=3859.18
step=36, loss=3794.43
step=37, loss=4585.11
step=38, loss=3592.34
step=39, loss=4383.95
step=40, loss=4529.52
step=41, loss=3980.33
step=42, loss=5718.45
step=43, loss=3567.15
step=44, loss=4701.89
step=45, loss=3430.48
step=46, loss=3573.

In [61]:
for x, y in  test_DataLoader:
    x=jnp.array(x)
    y=jnp.array(y)
    pred_y = jax.vmap(model)(x)
    mse = ((pred_y - y) ** 2).mean()
    print(f" mse={"%.2f" % mse},pred_y={"%.2f" % pred_y.item()}, y={"%.2f" % y.item()}")



 mse=4125.77,pred_y=0.22, y=-64.01
 mse=4277.64,pred_y=0.22, y=-65.18
 mse=4232.23,pred_y=0.22, y=-64.83
